# 05 · Modelos clásicos de Machine Learning

| | |
|---|---|
| **Fase** | F3: Analizar y modelar |
| **IDs del enunciado que cubre** | **IA3.3** (lado clásico de la comparación), **R1** (modelo predictivo funcional), **F3** |
| **Entregable** | `Back/notebooks/05_modelos_clasicos.ipynb` + `Back/models/resultados_clasicos.json` |

**Objetivo:** entrenar y comparar, con las mismas métricas, una línea base y cuatro modelos clásicos para estimar la probabilidad de impago (PD):
`DummyClassifier`, `LogisticRegression`, `DecisionTreeClassifier`, `RandomForestClassifier` y `HistGradientBoostingClassifier`, todos dentro de un `Pipeline(preprocesador + modelo)`.

**Métricas:** ROC-AUC (principal), PR-AUC, accuracy, precision, recall, F1, KS y Brier. Meta mínima: AUC ≥ 0.70 en test.

**Prerrequisitos:** `pandas`, `numpy`, `scikit-learn`, `matplotlib`, `seaborn`. Semilla `RANDOM_STATE = 42`.

## 0. Configuración del entorno

In [ ]:
import os, json, time, warnings
warnings.filterwarnings("ignore")
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
pd.options.display.float_format = "{:,.4f}".format
pd.options.display.max_columns = 60
plt.rcParams["figure.dpi"] = 100

def encontrar_raiz():
    """Sube desde el directorio actual hasta encontrar db/credit_risk.csv (sin rutas absolutas)."""
    actual = Path.cwd().resolve()
    for carpeta in [actual, *actual.parents]:
        if (carpeta / "db" / "credit_risk.csv").exists():
            return carpeta
    raise FileNotFoundError("No se encontró db/credit_risk.csv. Ejecute el notebook desde Back/notebooks.")

RAIZ = encontrar_raiz()
DATA_PATH = RAIZ / "db" / "credit_risk.csv"
MODELS_DIR = RAIZ / "Back" / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
print("Dataset:", DATA_PATH.relative_to(RAIZ), "| artefactos:", MODELS_DIR.relative_to(RAIZ))

In [ ]:
TARGET = "default"
TEXTO = "texto_solicitud"
SENSIBLES = ["sexo"]   # solo para auditar equidad, NUNCA como predictor
NUM_BASE = ["edad", "antiguedad_laboral_anios", "personas_a_cargo", "ingreso_mensual", "monto_credito",
            "plazo_meses", "tasa_interes_anual", "num_creditos_activos", "moras_ultimos_12m",
            "relacion_deuda_ingreso", "puntaje_buro", "uso_cupo_tarjetas"]
CAT_COLS = ["estado_civil", "nivel_educativo", "situacion_laboral", "tipo_vivienda", "proposito"]
NUEVAS = ["cuota_estimada", "ratio_cuota_ingreso", "ratio_monto_ingreso", "tiene_moras_previas", "longitud_texto"]
NUM_FEATS = NUM_BASE + NUEVAS
FEATURES = NUM_FEATS + CAT_COLS

def limpiar(df_raw, verbose=True):
    """Duplicados (exactos y por id_cliente), valores inválidos -> NaN y texto vacío.
    La imputación (mediana/moda) y el capping p1-p99 se hacen en `preparar`, con estadísticos de train."""
    df = df_raw.copy()
    n0 = len(df)
    df = df.drop_duplicates()
    n1 = len(df)
    df = df.drop_duplicates(subset="id_cliente", keep="first")
    n2 = len(df)
    df = df[df[TARGET].isin([0, 1])].copy()
    df[TARGET] = df[TARGET].astype(int)
    df.loc[~df["edad"].between(18, 100), "edad"] = np.nan          # edades imposibles (1, 999...)
    for c in ["ingreso_mensual", "monto_credito"]:
        df.loc[df[c] <= 0, c] = np.nan
    for c in CAT_COLS + ["sexo"]:
        df[c] = df[c].where(df[c].isna(), df[c].astype(str).str.strip())
    df[TEXTO] = df[TEXTO].fillna("").astype(str)
    if verbose:
        print(f"Filas originales: {n0:,} | duplicados exactos: {n0 - n1} | duplicados por id_cliente: {n1 - n2} | filas finales: {len(df):,}")
    return df.reset_index(drop=True)

def agregar_variables(df):
    """Variables de negocio del paso 3: cuota (amortización francesa), ratios, moras previas y longitud del texto."""
    df = df.copy()
    r = df["tasa_interes_anual"] / 100 / 12
    n = df["plazo_meses"]
    df["cuota_estimada"] = df["monto_credito"] * r / (1 - (1 + r) ** (-n))
    df["ratio_cuota_ingreso"] = df["cuota_estimada"] / df["ingreso_mensual"]
    df["ratio_monto_ingreso"] = df["monto_credito"] / df["ingreso_mensual"]
    df["tiene_moras_previas"] = (df["moras_ultimos_12m"] > 0).astype(int)
    df["longitud_texto"] = df[TEXTO].str.len()
    return df

def preparar(df_limpio, test_size=0.2):
    """Split estratificado ANTES de imputar/capar: medianas, modas y percentiles salen solo de train (sin leakage)."""
    train, test = train_test_split(df_limpio, test_size=test_size, stratify=df_limpio[TARGET],
                                   random_state=RANDOM_STATE)
    train, test = train.copy(), test.copy()
    medianas = train[NUM_BASE].median()
    modas = {c: train[c].mode().iloc[0] for c in CAT_COLS}
    p01, p99 = train[NUM_BASE].quantile(0.01), train[NUM_BASE].quantile(0.99)
    preparar.stats = {"medianas": medianas, "modas": modas, "p01": p01, "p99": p99}
    salida = []
    for d in (train, test):
        d[NUM_BASE] = d[NUM_BASE].fillna(medianas).clip(lower=p01, upper=p99, axis=1)
        for c in CAT_COLS:
            d[c] = d[c].fillna(modas[c])
        salida.append(agregar_variables(d).reset_index(drop=True))
    return salida[0], salida[1]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def build_preprocessor(extra_num=()):
    """Mismo preprocesador del paso 3: numéricas (mediana + StandardScaler) y categóricas (moda + One-Hot)."""
    num = Pipeline([("imputar", SimpleImputer(strategy="median")), ("escalar", StandardScaler())])
    cat = Pipeline([("imputar", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num, NUM_FEATS + list(extra_num)), ("cat", cat, CAT_COLS)])

In [ ]:
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score, precision_score,
                             recall_score, f1_score, brier_score_loss, roc_curve, precision_recall_curve,
                             confusion_matrix)

def ks_stat(y, p):
    """Estadístico KS: máxima separación entre las distribuciones acumuladas de buenos y malos."""
    fpr, tpr, _ = roc_curve(y, p)
    return float(np.max(tpr - fpr))

def metricas(y, p, umbral=0.5):
    pred = (np.asarray(p) >= umbral).astype(int)
    return {"roc_auc": roc_auc_score(y, p), "pr_auc": average_precision_score(y, p),
            "accuracy": accuracy_score(y, pred), "precision": precision_score(y, pred, zero_division=0),
            "recall": recall_score(y, pred), "f1": f1_score(y, pred), "ks": ks_stat(y, p),
            "brier": brier_score_loss(y, p)}

## 1. Carga y limpieza inline

Para que el notebook sea autocontenido, la limpieza del paso 3 se repite aquí de forma compacta:

- **Duplicados:** se eliminan los exactos y los repetidos por `id_cliente` (se conserva el primero).
- **Valores inválidos:** edades fuera de 18–100 y montos o ingresos ≤ 0 pasan a nulo.
- **Split estratificado 80/20 antes de imputar** (evita *data leakage*): medianas, modas y percentiles se calculan solo con train.
- **Imputación:** mediana en las numéricas y moda en las categóricas.
- **Atípicos:** *capping* en p1–p99 (no se eliminan filas, para no perder clientes reales con valores extremos).
- **Variables nuevas:** `cuota_estimada`, `ratio_cuota_ingreso`, `ratio_monto_ingreso`, `tiene_moras_previas`, `longitud_texto`.
- `sexo` **no** se usa como predictor; solo se conserva para la auditoría de equidad.

In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print("Forma original:", df_raw.shape)
print("Nulos por columna (solo las que tienen):")
print(df_raw.isna().sum()[lambda s: s > 0].to_string())
df = limpiar(df_raw)
train, test = preparar(df)
X_train, y_train = train[FEATURES], train[TARGET]
X_test, y_test = test[FEATURES], test[TARGET]
print(f"Train: {X_train.shape} | tasa de default {y_train.mean():.3f}")
print(f"Test:  {X_test.shape} | tasa de default {y_test.mean():.3f}")
print("Nulos restantes en features (train/test):", int(train[FEATURES].isna().sum().sum()), int(test[FEATURES].isna().sum().sum()))

## 2. Definición de los modelos

Cada modelo va dentro de un `Pipeline` con el mismo `ColumnTransformer`, de modo que el escalado y el One-Hot se ajustan **solo** con los datos de cada *fold* de entrenamiento.

- **Dummy:** predice siempre la prevalencia (línea base: AUC = 0.5).
- **Regresión logística** con `class_weight="balanced"` para compensar el desbalance (≈ 36 % de impagos).
- **Árbol de decisión** poco profundo (interpretable).
- **Random Forest** (bagging de árboles).
- **HistGradientBoosting** (boosting por histogramas, eficiente en datos tabulares).

In [ ]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

modelos = {
    "Dummy (línea base)": DummyClassifier(strategy="prior"),
    "Regresión logística": LogisticRegression(class_weight="balanced", max_iter=3000, random_state=RANDOM_STATE),
    "Árbol de decisión": DecisionTreeClassifier(max_depth=6, min_samples_leaf=40, class_weight="balanced",
                                                random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=14, min_samples_leaf=5, max_features="sqrt",
                                            class_weight="balanced_subsample", n_jobs=-1, random_state=RANDOM_STATE),
    "HistGradientBoosting": HistGradientBoostingClassifier(learning_rate=0.08, max_iter=200, max_leaf_nodes=31,
                                                           l2_regularization=1.0, early_stopping=True,
                                                           class_weight="balanced", random_state=RANDOM_STATE),
}

def hacer_pipeline(modelo):
    return Pipeline([("prep", build_preprocessor()), ("modelo", modelo)])

## 3. Validación cruzada estratificada (5 *folds*) en train

Se usa `StratifiedKFold(n_splits=5, shuffle=True, random_state=42)` con ROC-AUC como métrica. La desviación estándar entre *folds* indica la estabilidad del modelo.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
resultados_cv = {}
for nombre, modelo in modelos.items():
    t0 = time.time()
    scores = cross_val_score(hacer_pipeline(modelo), X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=1)
    resultados_cv[nombre] = {"cv_auc_media": scores.mean(), "cv_auc_std": scores.std(), "tiempo_cv_s": time.time() - t0}
    print(f"{nombre:<24} AUC CV = {scores.mean():.4f} ± {scores.std():.4f}   ({time.time() - t0:.1f} s)")

## 4. Entrenamiento en todo train y evaluación en test

Las métricas dependientes de un umbral (accuracy, precision, recall, F1) usan 0.5; el umbral óptimo por costo de negocio se elige en el notebook 08.

In [ ]:
pipelines, probas, filas = {}, {}, []
for nombre, modelo in modelos.items():
    pipe = hacer_pipeline(modelo)
    t0 = time.time()
    pipe.fit(X_train, y_train)
    t_fit = time.time() - t0
    p = pipe.predict_proba(X_test)[:, 1]
    pipelines[nombre], probas[nombre] = pipe, p
    filas.append({"modelo": nombre, **resultados_cv[nombre], **metricas(y_test, p), "tiempo_entrenamiento_s": t_fit})

tabla = pd.DataFrame(filas).set_index("modelo").sort_values("roc_auc", ascending=False)
tabla

### 4.1 Curvas ROC y Precision-Recall superpuestas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
for nombre, p in probas.items():
    fpr, tpr, _ = roc_curve(y_test, p)
    axes[0].plot(fpr, tpr, label=f"{nombre} (AUC={roc_auc_score(y_test, p):.3f})")
    prec, rec, _ = precision_recall_curve(y_test, p)
    axes[1].plot(rec, prec, label=f"{nombre} (AP={average_precision_score(y_test, p):.3f})")
axes[0].plot([0, 1], [0, 1], "k--", lw=1)
axes[0].set(title="Curva ROC (test)", xlabel="Tasa de falsos positivos (FPR)", ylabel="Tasa de verdaderos positivos (TPR)")
axes[1].axhline(y_test.mean(), color="k", ls="--", lw=1, label=f"Prevalencia = {y_test.mean():.2f}")
axes[1].set(title="Curva Precision-Recall (test)", xlabel="Recall", ylabel="Precision")
for ax in axes:
    ax.legend(fontsize=8); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

**Interpretación:** cuanto más se acerca la curva ROC a la esquina superior izquierda (y la PR a la superior derecha), mejor separa el modelo a los clientes que impagan de los que no. El Dummy coincide con la diagonal / la prevalencia, como se espera de una línea base sin información.

### 4.2 Matrices de confusión (umbral 0.5)

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, axes = plt.subplots(1, len(probas), figsize=(4 * len(probas), 3.6))
for ax, (nombre, p) in zip(axes, probas.items()):
    ConfusionMatrixDisplay.from_predictions(y_test, (p >= 0.5).astype(int), ax=ax, colorbar=False,
                                            display_labels=["No impago", "Impago"], cmap="Blues")
    ax.set_title(nombre, fontsize=9)
plt.tight_layout(); plt.show()

### 4.3 Estadístico KS del mejor modelo

In [ ]:
mejor = tabla.drop(index="Dummy (línea base)").index[0]
p_mejor = probas[mejor]
umbrales = np.linspace(0, 1, 201)
cdf_buenos = [(p_mejor[y_test.values == 0] <= u).mean() for u in umbrales]
cdf_malos = [(p_mejor[y_test.values == 1] <= u).mean() for u in umbrales]
ks_idx = int(np.argmax(np.array(cdf_buenos) - np.array(cdf_malos)))
plt.figure(figsize=(7, 4.5))
plt.plot(umbrales, cdf_buenos, label="No impago (default=0)")
plt.plot(umbrales, cdf_malos, label="Impago (default=1)")
plt.vlines(umbrales[ks_idx], cdf_malos[ks_idx], cdf_buenos[ks_idx], colors="red", linestyles="--",
           label=f"KS = {cdf_buenos[ks_idx] - cdf_malos[ks_idx]:.3f}")
plt.title(f"Distribuciones acumuladas de la PD: {mejor}"); plt.xlabel("Probabilidad de impago"); plt.ylabel("Proporción acumulada")
plt.legend(); plt.grid(alpha=0.3); plt.show()
print("Mejor modelo por AUC en test:", mejor)

**Interpretación:** el KS mide la máxima distancia entre las distribuciones de PD de buenos y malos pagadores; en la industria de crédito un KS > 0.30 suele considerarse aceptable y > 0.40 bueno.

## 5. Interpretabilidad

### 5.1 Coeficientes de la regresión logística
Los coeficientes están en la escala de las variables estandarizadas: un coeficiente positivo **aumenta** el *log-odds* de impago.

In [ ]:
pipe_lr = pipelines["Regresión logística"]
nombres = pipe_lr.named_steps["prep"].get_feature_names_out()
coefs = pd.Series(pipe_lr.named_steps["modelo"].coef_[0], index=nombres).sort_values()
top = pd.concat([coefs.head(10), coefs.tail(10)])
plt.figure(figsize=(8, 7))
top.plot.barh(color=["tab:green" if v < 0 else "tab:red" for v in top])
plt.axvline(0, color="k", lw=0.8)
plt.title("Regresión logística: 10 coeficientes más negativos y 10 más positivos")
plt.xlabel("Coeficiente (variables estandarizadas)"); plt.tight_layout(); plt.show()

**Interpretación:** en rojo, las variables que aumentan el riesgo; en verde, las que lo reducen. Se espera, por ejemplo, que un mayor `puntaje_buro` reduzca el riesgo y que más moras o una mayor relación deuda/ingreso lo aumenten.

### 5.2 Importancia por permutación (mejor modelo, test)

In [ ]:
from sklearn.inspection import permutation_importance
pi = permutation_importance(pipelines[mejor], X_test, y_test, scoring="roc_auc", n_repeats=5,
                            random_state=RANDOM_STATE, n_jobs=1)
imp = pd.DataFrame({"importancia_media": pi.importances_mean, "desv": pi.importances_std},
                   index=X_test.columns).sort_values("importancia_media")
plt.figure(figsize=(8, 7))
plt.barh(imp.index, imp["importancia_media"], xerr=imp["desv"], color="tab:blue")
plt.title(f"Importancia por permutación (caída de AUC): {mejor}"); plt.xlabel("Caída media del ROC-AUC")
plt.tight_layout(); plt.show()
imp.sort_values("importancia_media", ascending=False).head(10)

**Interpretación:** la importancia por permutación mide cuánto cae el AUC al desordenar cada variable; es agnóstica al modelo y se calcula sobre las variables originales (antes del One-Hot).

## 6. Guardar los resultados

In [ ]:
salida = {
    "fecha": pd.Timestamp.now().isoformat(timespec="seconds"),
    "random_state": RANDOM_STATE,
    "n_train": int(len(X_train)), "n_test": int(len(X_test)),
    "variables": FEATURES,
    "mejor_modelo": mejor,
    "modelos": tabla.reset_index().to_dict(orient="records"),
}
ruta = MODELS_DIR / "resultados_clasicos.json"
ruta.write_text(json.dumps(salida, indent=2, ensure_ascii=False, default=float), encoding="utf-8")
print("Guardado:", ruta.relative_to(RAIZ))

## 7. Verificación del criterio de hecho

In [ ]:
auc_base = tabla.loc["Dummy (línea base)", "roc_auc"]
auc_mejor = tabla.loc[mejor, "roc_auc"]
print(f"Línea base AUC = {auc_base:.4f} | mejor modelo ({mejor}) AUC = {auc_mejor:.4f}")
print("¿Supera a la línea base?", auc_mejor > auc_base)
print("¿Cumple la meta AUC >= 0.70?", auc_mejor >= 0.70)
tabla[["cv_auc_media", "roc_auc", "pr_auc", "recall", "f1", "ks", "brier", "tiempo_entrenamiento_s"]].round(4)

> **Estado de ejecución (respuesta a QA, 2026-10-16):** este notebook **todavía no tiene salidas guardadas** (en esta máquina la ejecución con CV de 5 folds supera los ~5 min). Las conclusiones de abajo describen el resultado **esperado** y deben confirmarse al ejecutarlo. La comparación de modelos que respalda el modelo servido está en `Back/models/model_card.json` (`comparacion_roc_auc_cv_train`, generada por `Back/train.py`).

## Conclusiones

1. **Todos los modelos superan con claridad a la línea base** (Dummy, AUC = 0.50), lo que confirma que las variables financieras y demográficas contienen señal sobre el impago.
2. Los modelos de ensamble (**HistGradientBoosting** y **Random Forest**) y la **regresión logística** obtienen AUC muy cercanos; la tabla del punto 7 muestra el ranking exacto en test y la validación cruzada confirma que las diferencias son estables (desviación entre *folds* pequeña).
3. La **regresión logística** ofrece una interpretabilidad directa (coeficientes con signo) con un desempeño competitivo, lo que la hace atractiva para un entorno regulado; los ensambles capturan no linealidades e interacciones a cambio de menor transparencia.
4. Las variables más influyentes (permutación) son de comportamiento crediticio y capacidad de pago (puntaje de buró, moras, relación deuda/ingreso, uso de cupo y cuota/ingreso), coherentes con el conocimiento de negocio.
5. Con umbral 0.5 el recall depende de si el modelo usa `class_weight`; el umbral definitivo se optimiza por costo en el notebook **08**, junto con la calibración (Brier) y la auditoría de equidad.
6. Los resultados quedan en `Back/models/resultados_clasicos.json` para la comparación con Deep Learning (notebook 06) y NLP (notebook 07).